## access the data 

In [0]:
%sql
select * from ecommerce_dwh.core.sales_trm limit 4

## create dimensions and handle the nulls

In [0]:
%sql
create table if not exists ecommerce_dwh.data_marts.dim_order
(

    dim_order_key int,
    order_id string,
    order_date date,
    payment_method string,
    order_status string
)

### dim_order

In [0]:
%sql
insert into ecommerce_dwh.data_marts.dim_order

select 
    row_number() over(order by o.order_id) as dim_order_key,
    o.order_id,
    o.order_date,
    o.payment_method,
    o.order_status
    from
      (
      select 
      distinct(order_id) as order_id,
      order_date,
      ifnull(payment_method,'Unknown') as payment_method,
      ifnull(order_status,'Unknown') as order_status 
      from ecommerce_dwh.core.sales_trm ) as o


### dim_customer

In [0]:
%sql
create table if not exists ecommerce_dwh.data_marts.dim_customer
(
    dim_customer_key int,
    customer_id string,
    customer_name string,
    city string
)

In [0]:
%sql
insert into ecommerce_dwh.data_marts.dim_customer
select 
      row_number() over(order by c.customer_id) as dim_customer_key,
      c.customer_id,
      c.customer_name,
      c.city
      from (
            select 
            distinct(customer_id) as customer_id,
            customer_name,
            ifnull(city,'Unknown') as city
            from ecommerce_dwh.core.sales_trm)
      as c

### dim_product

In [0]:
%sql
create table if not exists ecommerce_dwh.data_marts.dim_product(

    dim_product_key int,
    product_id string,
    product_name string,
    product_category string
    
)

In [0]:
%sql
insert into ecommerce_dwh.data_marts.dim_product
select 
      row_number() over(order by p.product_id) as dim_product_key,
      p.product_id,
      p.product_name,
      p.category
      from (
            select 
            distinct(product_id) as product_id,
            ifnull(product_name,'Unknown') as product_name,
            ifnull(category,'Unkonown') as
            category
            from ecommerce_dwh.core.sales_trm) as p

### dim_date_key

In [0]:
%sql
create table if not exists ecommerce_dwh.data_marts.dim_date(
    dim_date_key int,
    order_date date 
    )

In [0]:
%sql
insert into ecommerce_dwh.data_marts.dim_date
select 
      row_number() over(order by t.order_date) as dim_date_key,
      t.order_date
      from (
        select distinct(order_date) as order_date
        from ecommerce_dwh.core.sales_trm) as t

### fact table

In [0]:
%sql
create table if not exists ecommerce_dwh.data_marts.fact_table(
    order_id string,
    quantity int,
    unit_price double,
    total_price double,
    dim_customer_key int,
    dim_product_key int,
    dim_date_key int
)

In [0]:
%sql
insert into ecommerce_dwh.data_marts.fact_table
select 
        s.order_id,
        s.quantity,
        s.unit_price,
        s.quantity*s.unit_price as total_price,
        c.dim_customer_key,
        p.dim_product_key,
        d.dim_date_key
        from ecommerce_dwh.core.sales_trm as s
        left join 
            ecommerce_dwh.data_marts.dim_customer as c
            on s.customer_id = c.customer_id
        left join 
            ecommerce_dwh.data_marts.dim_product as p
            on s.product_id= p.product_id
        left join 
            ecommerce_dwh.data_marts.dim_date as d
            on s.order_date = d.order_date
        left join 
            ecommerce_dwh.data_marts.dim_order as o
            on s.order_id = o.order_id        
        

In [0]:
%sql
select * from ecommerce_dwh.data_marts.fact_table